# 00 - Data access and preprocessing

Capstone project 9: Credit Card Fraud Detection with Model Deployment.

In this notebook I load the Kaggle dataset, check that it is what I expect, and then
run `src/preprocess.py` which builds the train / validation / test files used by all the
other notebooks.

In [ ]:
import os
# the notebooks live in notebooks/, but all paths below are from the project root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())

## 1. Getting the dataset

Dataset: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud

```
kaggle datasets download -d mlg-ulb/creditcardfraud -p data/ --unzip
```

(or download the zip manually and put `creditcard.csv` inside `data/`). The file is not
committed to GitHub because it is about 150 MB.

In [ ]:
import pandas as pd

assert os.path.exists("data/creditcard.csv"), "Put creditcard.csv in the data/ folder first"
df = pd.read_csv("data/creditcard.csv")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

## 2. Validating the data

In [ ]:
counts = df["Class"].value_counts()
print(counts)
print("\nFraud rate: {:.4f}%".format(df["Class"].mean() * 100))
print("Missing values:", int(df.isnull().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Duplicates by class:", df[df.duplicated()]["Class"].value_counts().to_dict())

# the numbers I expect for this dataset
assert df.shape == (284807, 31)
assert counts[1] == 492

**What I found**

- 284,807 transactions and 31 columns: `Time`, `V1`-`V28`, `Amount`, `Class`.
- Only 492 frauds (0.173%), so the data is extremely imbalanced.
- No missing values, but 1,081 duplicate rows (19 of them fraud). If identical rows end up in
  both train and test the test score gets inflated, so I drop them before splitting.

## 3. Preprocessing decisions

| Decision | Choice | Reason |
|---|---|---|
| Duplicates | dropped (283,726 rows left, 473 fraud) | avoids train/test leakage |
| `Time` | dropped | it is seconds since the first transaction, a new transaction has no meaningful value for it |
| `Amount` | `log1p` then `StandardScaler` | very right skewed with large outliers |
| `V1`-`V28` | unchanged | already PCA outputs, the extreme values are the fraud signal so no clipping |
| Split | stratified 70 / 15 / 15 | keeps the same fraud rate in every split |
| Scaler | fit on train only | fitting on all data would leak information |
| SMOTE / class weights | not here | only allowed on the training set, done in the modelling notebooks |

In [ ]:
import runpy
runpy.run_path("src/preprocess.py")

## 4. Sanity checks on the saved splits

In [ ]:
import json

splits = {n: pd.read_csv(f"data/processed/{n}.csv") for n in ["train", "val", "test"]}
for n, d in splits.items():
    print(f"{n:5s} rows={len(d):>7}  fraud={int(d['Class'].sum()):>3}  fraud rate={d['Class'].mean()*100:.3f}%")

print("\nAmount in train after scaling:")
print(splits["train"]["Amount"].describe().round(3))
print("Missing values:", sum(int(d.isnull().sum().sum()) for d in splits.values()))
print("Feature order:", json.load(open("models/feature_order.json"))[:5], "...")

The fraud rate is the same (0.167%) in all three splits and `Amount` has mean about 0 and
std 1 on the training data. Validation is used for early stopping, model selection and the
threshold. **Test is only used once, at the very end.**